In [19]:
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-huggingface
!pip install -U langchain-chroma
!pip install -U sentence-transformers
!pip install -U chromadb
!pip install -U rank-bm25
!pip install -U tqdm
!pip install -U pypdf

In [20]:
from pathlib import Path
from google.colab import drive

Path('./content/drive').mkdir(parents=True, exist_ok=True)
drive.mount('./content/drive')

Drive already mounted at ./content/drive; to attempt to forcibly remount, call drive.mount("./content/drive", force_remount=True).


In [21]:
PDF_DIR  = "/content"
CHROMA_DB_PATH = "./chroma_db/rag_docs"
COLLECTION_NAME = "rag_docs"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 200
RESET_COLLECTION = False
TOP_K = 10
API_KEY = "Enter your API Key"
MODEL_NAME = "openai/gpt-4o-mini"
MAX_TOKENS = 4096

In [22]:
#Document Load
from pypdf import PdfReader
from langchain_core.documents import Document

pdf_dir = Path(PDF_DIR)
raw_documents = []

pdf_files = list(pdf_dir.glob('*.pdf'))

for pdf_file in pdf_files:
    try:
        reader = PdfReader(str(pdf_file))
        total_pages = len(reader.pages)
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if not text.strip():
                continue
            raw_documents.append(Document(
                    page_content = text,
                    metadata = {
                        "source" : pdf_file.name,
                        "page_num" : page_num + 1,
                        "total_pages" : total_pages
                    }

                )
            )
        print("Loaded Documents:" , pdf_file.name,"with pages of", total_pages)
    except Exception as E:
        print("Loaded document is failed due to", E)

    print("total pages loaded:", len(raw_documents))

Loaded Documents: 01-11-2020-205951Mindset by Carol S. Dweck.pdf with pages of 306
total pages loaded: 302
Loaded Documents: CCW332-Digital-Marketing-Lecture-Notes-1.pdf with pages of 268
total pages loaded: 570


In [23]:
raw_documents[0]

Document(metadata={'source': '01-11-2020-205951Mindset by Carol S. Dweck.pdf', 'page_num': 2, 'total_pages': 306}, page_content=' \n \nCarol Dweck is widely regarded as one of the world’s leading researchers\nin the fields of personality, social psychology and developmental\npsychology. She has been the William B. Ransford Professor of Psychology\nat Columbia University and is now the Lewis and Virginia Eaton Professor\nof Psychology at Stanford University and a member of the American\nAcademy of Arts and Sciences. Her scholarly book Self-Theories: Their\nRole in Motivation, Personality and Development was named Book of the\nYear by the World Education Fellowship. Her work has been featured in\nsuch publications as the New Yorker, Time, New York Times, Washington\nPost and Boston Globe. She lives with her husband in Palo Alto, California.')

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP
)

raw_chunks = splitter.split_documents(raw_documents)

def clean_chunk(text):
    text = str(text.encode('utf-8', errors = 'ignore').decode('utf-8'))
    return "".join(text.split())

chunks = []

for doc in raw_chunks:
    content = (doc.page_content or "").strip()
    if content:
        doc.page_content = clean_chunk(content)
        chunks.append(doc)

def _is_clean(text: str) -> bool:
    try:
        text.encode('utf-8')
        return True
    except Exception:
        return False

bad = [i for i,d in enumerate(chunks) if not _is_clean(d.page_content)]

print("Total chunks: ", len(raw_chunks))
print("Bad chunks: ", len(bad))
print("Cleaned chunks: ", len(chunks))

Total chunks:  1605
Bad chunks:  0
Cleaned chunks:  1605


In [25]:
chunks[105]

Document(metadata={'source': '01-11-2020-205951Mindset by Carol S. Dweck.pdf', 'page_num': 45, 'total_pages': 306}, page_content='somethingreallyamazing.Themoredepressedpeoplewiththegrowthmindsetfelt(shortofseveredepression),themoretheytookactiontoconfronttheirproblems,themoretheymadesuretokeepupwiththeirschoolwork,andthemoretheykeptupwiththeirlives.Theworsetheyfelt,themoredeterminedtheybecame!Infact,fromthewaytheyacted,itmighthavebeenhardtoknowhowdespondenttheywere.Hereisastoryayoungmantoldme.IwasafreshmananditwasthefirsttimeIhadbeenawayfromhome.Everyonewasastranger,thecourseswerehard,andastheyearworeonIfeltmoreandmoredepressed.Eventually,itreachedapointwhereIcouldhardlygetoutofbedinthemorning.ButeverydayIforcedmyselftogetup,shower,shave,anddowhateveritwasIneededtodo.OnedayIreallyhitalowpointandIdecidedtoaskforhelp,soIwenttotheteachingassistantinmypsychologycourseandaskedforheradvice.“Areyougoingtoyourclasses?”sheasked.“Yes,”Ireplied.')

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = EMBEDDING_MODEL
)

print("Model loaded Successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded Successfully


In [27]:
import shutil
from langchain_chroma import Chroma

chroma_path = Path(CHROMA_DB_PATH)

if RESET_COLLECTION and chroma_path.exists():
    shutil.rmtree(chroma_path)
    print("Existing collection deleted (RESET_COLLECTION=True)")

if chroma_path.exists() and not RESET_COLLECTION:
    vectorstore = Chroma(
        persist_directory=CHROMA_DB_PATH,
        embedding_function=embeddings,
        collection_name=COLLECTION_NAME,
    )
    print(f"Loaded existing vectorstore with ({vectorstore._collection.count()} chunks)")
else:
    if not chunks:
        raise RuntimeError("No chunks to embed. Load PDFs first.")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=CHROMA_DB_PATH,
        collection_name=COLLECTION_NAME,
    )
    print(f"New vectorstore created with ({vectorstore._collection.count()} chunks)")

Loaded existing vectorstore with (1605 chunks)


In [28]:
import requests
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any

class OpenRouterLLM(LLM):
    api_key:    str
    model:      str = MODEL_NAME
    max_tokens: int = MAX_TOKENS

    @property
    def _llm_type(self) -> str:
        return 'openrouter'

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> str:
        try:
            response = requests.post(
                url='https://openrouter.ai/api/v1/chat/completions',
                headers={
                    'Authorization': f'Bearer {self.api_key}',
                    'Content-Type':  'application/json',
                },
                json={
                    'model':      self.model,
                    'max_tokens': self.max_tokens,
                    'messages':   [{'role': 'user', 'content': prompt}],
                },
                timeout=60,   # prevent indefinite hangs
            )
        except requests.exceptions.Timeout:
            raise RuntimeError('OpenRouter request timed out after 60 s.')
        except requests.exceptions.ConnectionError as exc:
            raise RuntimeError(f'Network error reaching OpenRouter: {exc}') from exc

        if not response.ok:
            raise RuntimeError(
                f'OpenRouter API error {response.status_code}: {response.text}'
            )

        data = response.json()
        try:
            return data['choices'][0]['message']['content']
        except (KeyError, IndexError) as exc:
            raise RuntimeError(
                f'Unexpected OpenRouter response shape: {data}'
            ) from exc


llm = OpenRouterLLM(api_key=API_KEY)
print('OpenRouter LLM is now ready')
print(llm.invoke('Say hello in one sentence.'))

OpenRouter LLM is now ready
Hello! How can I assist you today?


In [29]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

PROMPT_TEMPLATE = """
You are an expert study assistant. Using ONLY the context provided below,
give a thorough and detailed answer to the question.

Your answer should:
- Be comprehensive and well-structured
- Use bullet points or numbered lists where appropriate
- Include relevant examples or explanations from the context
- Use headings to organise if the answer covers multiple topics
- Be as detailed as possible — do NOT summarise briefly
- If the answer is not in the context, say "I don't have enough information."

context : {context},

Question :{question}

Detailed Answer:"""

prompt = PromptTemplate(
    template = PROMPT_TEMPLATE,
    input_variable = ['context', 'question']
)

chroma_retriever = vectorstore.as_retriever(
    search_kwarags = {'K':TOP_K}
)

def format_docs(docs):
    parts = []
    for doc in docs:
        source = doc.metadata.get('source', 'unknown')
        page   = doc.metadata.get('page',   'N/A')
        parts.append(f'[Source: {source} | Page {page}]\n{doc.page_content}')
    return '\n\n'.join(parts)

qa_chain = (
    {
        'context':  chroma_retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print('QA chain ready')

QA chain ready


In [30]:
def ask(question: str) -> str:
    """Run a question through the RAG chain and print the result."""
    print(f"\n{'='*60}")
    print(f'❓ Question: {question}')
    print('='*60)

    try:
        answer = qa_chain.invoke(question)
    except RuntimeError as exc:
        print(f'❌ LLM error: {exc}')
        return ''

    print(f'\n💡 Answer:\n{answer}')

    docs = chroma_retriever.invoke(question)
    print(f'\n📚 Sources ({len(docs)} chunks):')
    seen = set()
    for doc in docs:
        source = doc.metadata.get('source', 'unknown')
        page   = doc.metadata.get('page',   'N/A')
        key    = (source, page)
        if key not in seen:
            seen.add(key)
            print(f'  • {source}  —  page {page}')

    return answer


# ── Try it ──
ask('Explain about Mindset.')


❓ Question: Explain about Mindset.

💡 Answer:
# Understanding Mindset

Mindset is a psychological framework that significantly influences how individuals perceive their abilities, challenges, and overall potential. The two primary types of mindsets identified are the *growth mindset* and the *fixed mindset*. Below, we will delve into these concepts, their implications, and examples derived from the provided context.

## 1. Growth Mindset

### Definition
- A growth mindset is the belief that abilities and intelligence can be developed through dedication, hard work, and learning from failures. 

### Characteristics
- **Adaptability and Resilience**: Individuals with a growth mindset see challenges as opportunities to learn and improve.
- **Positive Responses to Feedback**: They are more open to criticism and use it constructively.
- **Increased Motivation**: They are inspired to put in more effort and strive for higher achievements.

### Example from Context
- In the provided context, s

"# Understanding Mindset\n\nMindset is a psychological framework that significantly influences how individuals perceive their abilities, challenges, and overall potential. The two primary types of mindsets identified are the *growth mindset* and the *fixed mindset*. Below, we will delve into these concepts, their implications, and examples derived from the provided context.\n\n## 1. Growth Mindset\n\n### Definition\n- A growth mindset is the belief that abilities and intelligence can be developed through dedication, hard work, and learning from failures. \n\n### Characteristics\n- **Adaptability and Resilience**: Individuals with a growth mindset see challenges as opportunities to learn and improve.\n- **Positive Responses to Feedback**: They are more open to criticism and use it constructively.\n- **Increased Motivation**: They are inspired to put in more effort and strive for higher achievements.\n\n### Example from Context\n- In the provided context, students who participated in a g